# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = json.loads(dataset.metadata.json())
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @ids
print('Available Record Sets:')
record_sets_metadata = []
for rset in dataset.record_sets:
    print(f"- @id: {rset['@id']}, name: {rset.get('name', '(no name)')}")
    record_sets_metadata.append(rset)

print('\nFields for each record set:')
for rset in record_sets_metadata:
    print(f"\nRecord Set @id: {rset['@id']}")
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', None)
            field_name = field.get('name', None)
        else:
            field_id = field
            field_name = None
        print(f"  - Field @id: {field_id}, name: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we'll extract data from all user record sets.
# To demonstrate, use the main survey record set (by @id), most likely named 'main', 'survey', or similar.

# List all record sets' @ids
record_set_ids = [rset['@id'] for rset in dataset.record_sets]
print('Record Set @ids:', record_set_ids)

# Prepare dictionary of DataFrames
dataframes = {}

# Load each record set (Limit records to avoid large memory use on huge sets)
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    dataframes[rset_id] = pd.DataFrame(records)

# We will focus on the primary record set for further analysis (assume first one as main for this notebook)
main_record_set_id = record_set_ids[0]
print(f'Columns for record set @id: {main_record_set_id}')
print(dataframes[main_record_set_id].columns.tolist())
# Display the first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Select a numeric field for filtering/normalization. We'll search for 'GAD-7' or 'PHQ-9' score field.
import numpy as np
df = dataframes[main_record_set_id]

# Try to find a likely numeric column for demonstration
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_columns:
    # If none found, try to convert possible columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

# If 'gad7_score' in columns, use it, else pick first numeric field
if 'gad7_score' in df.columns:
    numeric_field = 'gad7_score'
elif numeric_columns:
    numeric_field = numeric_columns[0]
else:
    numeric_field = df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field}")

# Filter, normalize, and group
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field, e.g., 'gender' or 'marital_status' if present
group_fields = ['gender', 'marital_status', 'age_group', 'level_of_education']
group_field = next((col for col in group_fields if col in df.columns), None)
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print('No suitable group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Optional: Boxplot by group field, if available
if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded the Croissant schema for the Kilifi County mental health survey, examined the record sets and fields, extracted the primary survey responses, and performed basic exploratory data analysis. We demonstrated simple filtering, normalization, grouping, and generated visualizations to inspect data distributions and group-wise effects. Further analysis could explore relationships between psychological and demographic fields, missingness, and more advanced statistical techniques!*